## Hyperparameter tuning

For this task we are testing the tool Optuna, which runs several training trials with various options and keeps the best performing model based on the metric of choice. It shall be set to optimize the roc_auc_score

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import optuna
import traceback
import lightgbm as lgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings("ignore")

In [11]:
# Helper: Clean LightGBM column names
def clean_feature_names(df):
    df.columns = df.columns.str.replace(r'[^A-Za-z0-9_]+', '_', regex=True)
    return df

# Helper: Fix CatBoost NaNs
def fix_catboost_categories(df, categorical_features):
    df_fixed = df.copy()
    for c in categorical_features:
        df_fixed[c] = df_fixed[c].astype(str).fillna("missing")
    return df_fixed

In [12]:
# Set paths and load raw as well as encoded datasets

ROOT = Path.cwd().parent # Path is anchored relative to this notebook location

DATA = ROOT / "data"

train_encoded = pd.read_csv(DATA / "processed" / "training_fe_full.csv", index_col="respondent_id")
test_encoded = pd.read_csv(DATA / "processed" / "test_fe_full.csv", index_col="respondent_id")

train_ft_raw = pd.read_csv(DATA / "raw" / "training_set_features.csv", index_col="respondent_id")
train_lb_raw = pd.read_csv(DATA / "raw" / "training_set_labels.csv", index_col="respondent_id")
test_raw = pd.read_csv(DATA / "raw" / "test_set_features.csv", index_col="respondent_id")

# Merge raw data into one single dataframe
train_raw = train_ft_raw.merge(train_lb_raw, left_index=True, right_index=True)

In [13]:
# Define targets and features
target_cols = ['h1n1_vaccine', 'seasonal_vaccine']
categorical_features = train_ft_raw.select_dtypes(include=['object', 'category']).columns.tolist()

X_raw = train_ft_raw
y1 = train_lb_raw['h1n1_vaccine']
y2 = train_lb_raw['seasonal_vaccine']

In [19]:
# Optuna function
def objective(trial, verbose=False):
    model_choice = "lightgbm"

    # Split setup
    kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    roc_scores = []

    # LightGBM params
    lgb_params = {
        "n_estimators": trial.suggest_int("lgb_n_estimators", 200, 1000),
        "learning_rate": trial.suggest_float("lgb_learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("lgb_num_leaves", 16, 64),
        "max_depth": trial.suggest_int("lgb_max_depth", 4, 10),
        "subsample": trial.suggest_float("lgb_subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("lgb_colsample_bytree", 0.5, 1.0),
        "random_state": 42
    }

    # TQDM progress bar for CV
    pbar = tqdm(total=kf.get_n_splits(), disable=not verbose, desc=f"{model_choice} CV")

    for fold, (train_idx, valid_idx) in enumerate(kf.split(train_ft_raw, train_lb_raw["h1n1_vaccine"])):
        try:
            X_train_raw = train_ft_raw.iloc[train_idx]
            X_valid_raw = train_ft_raw.iloc[valid_idx]

            y1_train = train_lb_raw["h1n1_vaccine"].iloc[train_idx]
            y1_valid = train_lb_raw["h1n1_vaccine"].iloc[valid_idx]

            y2_train = train_lb_raw["seasonal_vaccine"].iloc[train_idx]
            y2_valid = train_lb_raw["seasonal_vaccine"].iloc[valid_idx]

            if model_choice == "lightgbm":
                try:
                    X_train = train_encoded.loc[X_train_raw.index]
                    X_valid = train_encoded.loc[X_valid_raw.index]
                except Exception:
                    X_train = pd.get_dummies(X_train_raw, drop_first=True)
                    X_valid = pd.get_dummies(X_valid_raw, drop_first=True)
                    X_train, X_valid = X_train.align(X_valid, join="left", axis=1, fill_value=0)

                X_train = clean_feature_names(X_train)
                X_valid = clean_feature_names(X_valid)

                model1 = lgb.LGBMClassifier(**lgb_params)
                model2 = lgb.LGBMClassifier(**lgb_params)

                model1.fit(X_train, y1_train)
                model2.fit(X_train, y2_train)

                preds1 = model1.predict_proba(X_valid)[:, 1]
                preds2 = model2.predict_proba(X_valid)[:, 1]

            # Compute mean ROC-AUC across both targets
            auc1 = roc_auc_score(y1_valid, preds1)
            auc2 = roc_auc_score(y2_valid, preds2)
            roc_scores.append(np.mean([auc1, auc2]))

        except Exception as e:
            if verbose:
                print(f"❌ Error in fold {fold+1}: {e}")
            roc_scores.append(0.0)

        pbar.update(1)

    pbar.close()

    mean_auc = np.mean(roc_scores)
    if verbose:
        print(f"Trial done | Mean ROC-AUC: {mean_auc:.4f} | Model: {model_choice}")

    return mean_auc


In [20]:
# Initialize study
optimization_study_1 = optuna.create_study(direction="maximize")
optimization_study_1.optimize(lambda trial: objective(trial, verbose=False), n_trials=10)

print("Best trial:")
print(optimization_study_1.best_trial.params)
print("Best mean CV ROC-AUC:", optimization_study_1.best_value)

[I 2025-11-11 20:56:49,164] A new study created in memory with name: no-name-49452587-5fd8-4e14-810b-32a2c80df80a


[LightGBM] [Info] Number of positive: 3782, number of negative: 14022
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004550 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1048
[LightGBM] [Info] Number of data points in the train set: 17804, number of used features: 63
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.212424 -> initscore=-1.310375
[LightGBM] [Info] Start training from score -1.310375
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

[I 2025-11-11 20:57:03,271] Trial 0 finished with value: 0.8558520448291235 and parameters: {'lgb_n_estimators': 779, 'lgb_learning_rate': 0.05246774637428669, 'lgb_num_leaves': 23, 'lgb_max_depth': 6, 'lgb_subsample': 0.9941808417143743, 'lgb_colsample_bytree': 0.9269563460996829}. Best is trial 0 with value: 0.8558520448291235.


[LightGBM] [Info] Number of positive: 3782, number of negative: 14022
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003612 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1048
[LightGBM] [Info] Number of data points in the train set: 17804, number of used features: 63
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.212424 -> initscore=-1.310375
[LightGBM] [Info] Start training from score -1.310375
[LightGBM] [Info] Number of positive: 8264, number of negative: 9540
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003750 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1048
[LightGBM] [Info] Number of data points in the train set: 17804, number of used features: 63
[LightGBM] [Info] [bi

[I 2025-11-11 20:57:17,237] Trial 1 finished with value: 0.8633397911221868 and parameters: {'lgb_n_estimators': 889, 'lgb_learning_rate': 0.013886422212908553, 'lgb_num_leaves': 17, 'lgb_max_depth': 9, 'lgb_subsample': 0.8691937184769476, 'lgb_colsample_bytree': 0.9930066923323599}. Best is trial 1 with value: 0.8633397911221868.


...

[LightGBM] [Info] Number of positive: 3782, number of negative: 14022
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003230 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1048
[LightGBM] [Info] Number of data points in the train set: 17804, number of used features: 63
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.212424 -> initscore=-1.310375
[LightGBM] [Info] Start training from score -1.310375
[LightGBM] [Info] Number of positive: 8264, number of negative: 9540
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003791 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1048
[LightGBM] [Info] Number of data points in the train set: 17804, number of used features: 63
[LightGBM] [Info] [bi

[I 2025-11-11 20:58:48,957] Trial 8 finished with value: 0.8631597882844867 and parameters: {'lgb_n_estimators': 300, 'lgb_learning_rate': 0.03806162703824817, 'lgb_num_leaves': 33, 'lgb_max_depth': 10, 'lgb_subsample': 0.5984455079445445, 'lgb_colsample_bytree': 0.7455412517603605}. Best is trial 4 with value: 0.864572012166586.


[LightGBM] [Info] Number of positive: 3782, number of negative: 14022
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004997 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1048
[LightGBM] [Info] Number of data points in the train set: 17804, number of used features: 63
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.212424 -> initscore=-1.310375
[LightGBM] [Info] Start training from score -1.310375
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

[I 2025-11-11 20:59:01,038] Trial 9 finished with value: 0.864331911870361 and parameters: {'lgb_n_estimators': 784, 'lgb_learning_rate': 0.016958998281907327, 'lgb_num_leaves': 62, 'lgb_max_depth': 4, 'lgb_subsample': 0.5620148747232855, 'lgb_colsample_bytree': 0.7166534447921302}. Best is trial 4 with value: 0.864572012166586.


Best trial:
{'lgb_n_estimators': 325, 'lgb_learning_rate': 0.014252929520878387, 'lgb_num_leaves': 49, 'lgb_max_depth': 9, 'lgb_subsample': 0.7050140145384993, 'lgb_colsample_bytree': 0.5271084928217322}
Best mean CV ROC-AUC: 0.864572012166586


## Results

After Optimization, the best trial had the parameters:

Model: LightGBM

Best trial:
{'lgb_n_estimators': 325, 'lgb_learning_rate': 0.014252929520878387, 'lgb_num_leaves': 49, 'lgb_max_depth': 9, 'lgb_subsample': 0.7050140145384993, 'lgb_colsample_bytree': 0.5271084928217322}

Best mean CV ROC-AUC: 0.864572012166586

- CatBoost didn't do as good and also led to crashes so I removed it from optuna study
- Objective function could still be optimized
- Winning model can be retrained as final model and tested on other metrics as well (F1)